# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepalsr/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

1. Ranking signal analysis
Which safe content and search signals are associated with visibility, clicks, engagement, or movement?

Why?
- The starter data already has the signals this lane needs: impressions, clicks, CTR, average position, sessions, engagement and scroll rate, word count, page age, and the trend bucket.
- It is the most honest place to start. Before I can rank pages for review (Lane 2) or score CTR gaps (Lane 4), I need to know which signals carry information and how big those effects are.
- It keeps me disciplined about the difference between "associated with" and "causes". The data is observational.
- The result feeds every other lane. If I change lanes by Week 4, the signal audit is not wasted.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Search question.** Among pages that already get search impressions, which observable signals (position, volume, page age, length, engagement) are most strongly associated with clicks and with a page losing visibility, after accounting for position and volume?

**Unit of analysis.** One page (`content_id`) over the starter data's 90-day window. Rows with no impressions or younger than 90 days are excluded, as in the starter pipeline.

**Output.** A signal report: for each signal, an effect size (how big, not just "is there a link"), a chart, and a short plain-language recommendation. Plus a list of the caveats.

**Decision it improves.** Which page attributes a content team should look at first when deciding where limited review time goes.

**Action someone could take.** An editor or content strategist uses the report to decide which page types to review first, for example "check pages that sit on page one but have weak CTR" or "check older pages with high impressions", instead of scanning the whole inventory by hand. The action is *review*, not *rewrite*.

**Cost of a wrong recommendation.**

- *Wasted reviewer time:* a reviewer looks at pages that did not need attention.
- *Wrong lesson:* the team standardises on a signal that is only a correlation (for example, "make every article longer") and spends effort with no payoff.
- *False confidence:* a page is left alone because its signals looked fine, while it was quietly declining.

The first is cheap and recoverable. The second and third are the real costs, which is why the report must give effect sizes and caveats, not just a list of "important features".

**Why data or ML can help at all.** The inventory is far too large to read page by page, and the raw signals are entangled: high-impression pages tend to sit in better positions, and better positions earn more clicks. Grouped summaries, correlations, and simple models can separate those effects and show where a signal still matters after adjusting for the others.

**Why this is not just "train a model".** The deliverable is an *explanation of which signals matter and by how much*, not a prediction engine. Any model I use (for example, feature importance from a simple one) is a tool for reading the data, not the product. The starter target (`trend_direction == "down"`) is only a proxy computed from the current window, not a future outcome, so I will not treat a good score on it as proof of anything.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

CANDIDATES = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, "Starter CSV not found. Run from work/notebooks/ or the repo root."

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}: {len(raw):,} rows x {raw.shape[1]} columns")

# Same filters as the starter pipeline (scripts/01_prepare_features.py)
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset="content_id")

print(f"After starter filters: {len(df):,} pages across {df['client_id'].nunique():,} clients")

Loaded data/raw/content_refresh_anonymized.csv: 30,000 rows x 44 columns
After starter filters: 30,000 pages across 32 clients


In [7]:
is_down = df["trend_direction"].eq("down")
n_down = int(is_down.sum())
decline_rate = is_down.mean()
print(f"Pages with trend_direction == 'down': {n_down:,} of {len(df):,} ({decline_rate:.1%})")
print(df["trend_direction"].value_counts(normalize=True).round(3).to_string())

Pages with trend_direction == 'down': 16,262 of 30,000 (54.2%)
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038


In [8]:
MIN_IMPRESSIONS = 100
vis = df[df["impressions_90d"] >= MIN_IMPRESSIONS]

ctr_by_pos = (
    vis.groupby("position_tier")["ctr"]
       .agg(pages="count", median_ctr="median")
       .sort_values("median_ctr", ascending=False)
)
print(f"Pages with >= {MIN_IMPRESSIONS} impressions: {len(vis):,}")
print(ctr_by_pos.round(3).to_string())

hi, lo = ctr_by_pos["median_ctr"].iloc[0], ctr_by_pos["median_ctr"].iloc[-1]
if lo > 0:
    print(f"\nMedian CTR spans {lo:.3f} to {hi:.3f} across position tiers ({hi / lo:.1f}x gap)")
else:
    print("\nLowest tier median CTR is 0; see table.")

Pages with >= 100 impressions: 22,006
               pages  median_ctr
position_tier                   
page_1          8633        0.23
top_3            533        0.19
striking        5903        0.15
page_3_5        6058        0.06
deep             879        0.00

Lowest tier median CTR is 0; see table.


In [9]:
GROUP_COL = "age_tier" if "age_tier" in df.columns else "content_age_days"
if GROUP_COL == "content_age_days":
    df["_age_bucket"] = pd.qcut(df["content_age_days"], 4, duplicates="drop")
    GROUP_COL = "_age_bucket"

decline_by_group = (
    df.assign(is_down=is_down)
      .groupby(GROUP_COL, observed=True)["is_down"]
      .agg(pages="count", decline_rate="mean")
)
print(decline_by_group.round(3).to_string())

gap = decline_by_group["decline_rate"].max() - decline_by_group["decline_rate"].min()
print(f"\nDecline rate ranges over {gap * 100:.1f} percentage points across groups "
      f"(overall: {decline_rate:.1%}).")

          pages  decline_rate
age_tier                     
181-365   11368         0.515
31-90       492         0.669
365+       6360         0.426
91-180    11780         0.626

Decline rate ranges over 24.2 percentage points across groups (overall: 54.2%).


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

**I can claim:**
- In the starter sample, 54.2% of pages (16,262 of 30,000) are labelled "down" by the dataset's trend bucket.
- Median CTR is lower for pages in deeper position tiers: 0.23 on page 1, 0.15 in "striking", 0.06 on pages 3-5, and 0.00 in "deep" (pages with at least 100 impressions).
- The share of "down" pages differs by age group, from 66.9% (31-90 days) to 42.6% (365+ days), a gap of 24.2 percentage points.
- These patterns are *associated with* position and age, and they *suggest* signals worth studying further.

**I can't claim:**
- That page age or position *causes* a page to decline or to get clicks. This is observational data, so I only watched what happened and ran no experiment.
- That 54.2% of pages have truly declined. "Down" is a bucket from the current window and may also reflect seasonality, consolidation, or noise.
- That I found a Google ranking factor.
- That refreshing a page will make it recover.
- That `top_3` gets fewer clicks than `page_1`. That tier has only 533 pages, so the gap may just reflect small samples or the mix of pages.
- Anything about AI citations or AI rankings.

**Limits of this first look:**
- It uses a 30,000-row starter slice, not the full warehouse, so nothing here is a benchmark.
- Position, volume, and age are entangled, so single-variable summaries can mislead until I adjust for them.
- I haven't yet verified how the age groups were defined (a `31-90` group appeared even though I filtered for pages 90 days or older).
- No client names, URLs, queries, or titles are shown or reconstructed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.